# Tsukumo -- Monte Carlo What-If Scenario Engine (5 Models)

> This notebook implements a **Synthetic Data Generation (SDG) + Monte Carlo Simulation** pipeline. It:
> 1. Synthesises thousands of *virtual versions* of a patient using **Framingham & NHANES** statistical priors
> 2. Injects stochastic lifestyle shocks
> 3. Runs each virtual cohort through **all 5 models**: cardiac-arrest, diabetes, burnout, respiratory, kidney stones
> 4. Produces **probability distributions** with 80/95% confidence intervals and projected 6-month trajectories

---
**References**
- Framingham Heart Study: D'Agostino et al. (2008), NHANES 2017-2020
- Simulation method: Monte Carlo Methods in Risk Analysis


In [ ]:
# ── 0. Install / check dependencies ──────────────────────────────────────────
import subprocess, sys

REQUIRED = [
    "numpy", "pandas", "scipy", "scikit-learn",
    "matplotlib", "seaborn", "joblib", "tqdm", "ipywidgets"
]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"  Installing {pkg}…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅  All dependencies satisfied.")

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import Dict, Optional, List

warnings.filterwarnings("ignore")

# ── Plotting aesthetics ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#16162a",
    "axes.edgecolor":   "#2a2a4a",
    "axes.labelcolor":  "#c8c8e8",
    "axes.titlecolor":  "#e8e8ff",
    "axes.titlesize":   14,
    "axes.labelsize":   11,
    "xtick.color":      "#8888aa",
    "ytick.color":      "#8888aa",
    "text.color":       "#c8c8e8",
    "grid.color":       "#2a2a4a",
    "grid.linestyle":   "--",
    "legend.facecolor": "#1e1e38",
    "legend.edgecolor": "#3a3a6a",
    "font.family":      "DejaVu Sans",
})

PALETTE = {
    "baseline": "#7b7bff",
    "best":     "#2ecc71",
    "worst":    "#e74c3c",
    "moderate": "#f39c12",
    "fill_lo":  "#2ecc711a",
    "fill_hi":  "#e74c3c1a",
}

print("✅  Imports complete.")

---
## 🧱 Section 1 — Framingham / NHANES Statistical Priors

These distributions encode the **population-level** statistics for adult Americans (NHANES 2017-2020) and the Framingham Heart Study cohort.  
Every synthetic patient will be drawn from these priors and then *conditioned* on the real user's baseline measurements.

In [ ]:
# ── 2. Population Priors  ─────────────────────────────────────────────────────
# Format: { feature: (mean, std, low_clip, high_clip) }
#
# Sources:
#   NHANES 2017-2020: https://wwwn.cdc.gov/nchs/nhanes/
#   Framingham Heart Study: D'Agostino et al. (2008) Circ 117:743-753

NHANES_PRIORS: Dict[str, tuple] = {
    # ── Demographics ──────────────────────────────────────────────────────────
    "age":                       (47.8,  18.0,  18,  90),
    "bmi":                       (29.6,   7.0,  15,  60),

    # ── Cardiovascular ────────────────────────────────────────────────────────
    "sysBP":                     (124.0, 19.0,  80, 220),   # Systolic BP (mmHg)
    "diaBP":                     ( 76.0, 12.0,  50, 130),   # Diastolic BP (mmHg)
    "heartRate":                 ( 73.0, 13.0,  40, 140),   # BPM
    "totChol":                   (197.0, 40.0, 100, 400),   # mg/dL
    "HDLChol":                   ( 54.0, 16.0,  20, 110),   # mg/dL
    "glucose":                   (104.0, 30.0,  60, 400),   # mg/dL fasting

    # ── Lifestyle ─────────────────────────────────────────────────────────────
    "cigarettes_per_day":        (  3.5,  7.0,   0,  60),   # avg incl. non-smokers
    "sleep_hours":               (  6.9,  1.4,   3,  12),
    "physical_activity_met":     (  2.8,  1.6,   1,  12),   # MET-hrs/week
    "sodium_mg_per_day":         (3440,  900,  500, 8000),
    "alcohol_drinks_per_week":   (  3.1,  5.5,   0,  50),

    # ── Metabolic ─────────────────────────────────────────────────────────────
    "prevalentHyp":              (  0.45,  0.497, 0, 1),    # 0/1 binary
    "diabetes":                  (  0.11,  0.31,  0, 1),
    "currentSmoker":             (  0.14,  0.35,  0, 1),
}

# Binary feature names (clamped to 0/1 after sampling)
BINARY_FEATURES = {"prevalentHyp", "diabetes", "currentSmoker"}

print(f"✅  {len(NHANES_PRIORS)} population priors loaded.")

---
## Section 2 -- Model Loader

We load **all 5** pkl models used by the Flask micro-service:
`cardiac-arrest.pkl`, `diabetes.pkl`, `burnout_risk_model.pkl`, `respiratory_model.pkl`, `rf_kidney_stones.pkl`.
If any model is absent, a logistic-regression fallback is trained on synthetic data.


In [ ]:
# -- 3. Model Loader ----------------------------------------------------------
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
PUBLIC_DIR   = os.path.join(PROJECT_ROOT, 'public')

def load_pkl(path):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            model = pickle.load(f)
        print(f'  OK  Loaded: {os.path.basename(path)}')
        return model
    print(f'  WARN  Not found: {path} -> using synthetic fallback')
    return None

cardiac_model     = load_pkl(os.path.join(PUBLIC_DIR, 'cardiac-arrest.pkl'))
diabetes_model    = load_pkl(os.path.join(PUBLIC_DIR, 'diabetes.pkl'))
burnout_model     = load_pkl(os.path.join(PUBLIC_DIR, 'burnout_risk_model.pkl'))
respiratory_model = load_pkl(os.path.join(PUBLIC_DIR, 'respiratory_model.pkl'))
kidney_model      = load_pkl(os.path.join(PUBLIC_DIR, 'rf_kidney_stones.pkl'))

# -- Synthetic fallback -------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def _build_fallback_model(feature_names, seed=42):
    rng = np.random.default_rng(seed)
    n   = 5_000
    X   = rng.standard_normal((n, len(feature_names)))
    logit = np.full(n, -1.5)
    for nm, coef in [('sysBP', 0.03), ('totChol', 0.02), ('age', 0.04),
                     ('bmi', 0.02), ('glucose', 0.02)]:
        if nm in feature_names:
            logit += coef * X[:, feature_names.index(nm)]
    y = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
    pipe.fit(X, y)
    return pipe

# -- Feature lists ------------------------------------------------------------
CARDIAC_FEATURES = [
    'age', 'sysBP', 'diaBP', 'heartRate', 'totChol', 'HDLChol',
    'glucose', 'bmi', 'currentSmoker', 'cigarettes_per_day', 'prevalentHyp'
]
DIABETES_FEATURES = [
    'age', 'bmi', 'glucose', 'sysBP', 'diaBP', 'totChol',
    'HDLChol', 'sleep_hours', 'physical_activity_met', 'sodium_mg_per_day'
]
BURNOUT_FEATURES = [
    'age', 'bmi', 'sleep_hours', 'physical_activity_met',
    'sodium_mg_per_day', 'alcohol_drinks_per_week',
    'sysBP', 'heartRate', 'glucose'
]
RESPIRATORY_FEATURES = [
    'age', 'bmi', 'sysBP', 'diaBP', 'heartRate',
    'currentSmoker', 'cigarettes_per_day', 'prevalentHyp',
    'glucose', 'totChol'
]
KIDNEY_FEATURES = [
    'age', 'bmi', 'glucose', 'sysBP', 'diaBP',
    'sodium_mg_per_day', 'totChol', 'HDLChol',
    'physical_activity_met', 'alcohol_drinks_per_week'
]

if cardiac_model     is None:
    cardiac_model     = _build_fallback_model(CARDIAC_FEATURES,     seed=1)
    print('  Fallback built: cardiac')
if diabetes_model    is None:
    diabetes_model    = _build_fallback_model(DIABETES_FEATURES,    seed=2)
    print('  Fallback built: diabetes')
if burnout_model     is None:
    burnout_model     = _build_fallback_model(BURNOUT_FEATURES,     seed=3)
    print('  Fallback built: burnout')
if respiratory_model is None:
    respiratory_model = _build_fallback_model(RESPIRATORY_FEATURES, seed=4)
    print('  Fallback built: respiratory')
if kidney_model      is None:
    kidney_model      = _build_fallback_model(KIDNEY_FEATURES,      seed=5)
    print('  Fallback built: kidney')

MODEL_REGISTRY = [
    ('cardiac',     cardiac_model,     CARDIAC_FEATURES),
    ('diabetes',    diabetes_model,    DIABETES_FEATURES),
    ('burnout',     burnout_model,     BURNOUT_FEATURES),
    ('respiratory', respiratory_model, RESPIRATORY_FEATURES),
    ('kidney',      kidney_model,      KIDNEY_FEATURES),
]

print()
print('OK  All 5 models ready:')
for name, _, feats in MODEL_REGISTRY:
    print(f'   * {name:12s}  ({len(feats)} features)')

---
## 🧪 Section 3 — Synthetic Data Generation (SDG)

We generate a virtual **cohort** of `N_SIMULATIONS` patients.  
Each patient is drawn from the NHANES/Framingham multivariate normal prior, then **conditioned** so the population mean equals the real user's measured values. This is equivalent to a **Bayesian update** of the population prior given observed data.

In [ ]:
# ── 4. SDG Engine ─────────────────────────────────────────────────────────────

@dataclass
class PatientProfile:
    """Real observed measurements for the user — edit to match your patient."""
    # ── Demographics ──────────────────────────────────────────────────────────
    age:                     float = 45.0
    bmi:                     float = 28.5

    # ── Cardiovascular ────────────────────────────────────────────────────────
    sysBP:                   float = 140.0
    diaBP:                   float = 90.0
    heartRate:               float = 85.0
    totChol:                 float = 215.0
    HDLChol:                 float = 45.0
    glucose:                 float = 160.0

    # ── Lifestyle ─────────────────────────────────────────────────────────────
    cigarettes_per_day:      float = 0.0
    sleep_hours:             float = 6.0
    physical_activity_met:   float = 1.5
    sodium_mg_per_day:       float = 3800.0
    alcohol_drinks_per_week: float = 4.0

    # ── Metabolic flags ───────────────────────────────────────────────────────
    prevalentHyp:            float = 1.0
    diabetes:                float = 0.0
    currentSmoker:           float = 0.0


def generate_cohort(
    profile: PatientProfile,
    n: int = 10_000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Draw n synthetic patients from NHANES priors, then shift the distribution
    so it is centred on the real patient's observed measurements.

    This implements a "Bayesian personalisation" step: the population spread
    (variance) is preserved but the mean is pulled to the individual's values.
    """
    rng = np.random.default_rng(seed)
    rows = {}

    for feat, (pop_mean, pop_std, lo, hi) in NHANES_PRIORS.items():
        # Draw from population
        samples = rng.normal(loc=pop_mean, scale=pop_std, size=n)

        # Shift centre to user's value (conditional mean update)
        user_val = getattr(profile, feat, pop_mean)
        delta    = user_val - pop_mean
        samples  = samples + delta

        # Clip to physiologically plausible range
        samples  = np.clip(samples, lo, hi)

        # Binarise if needed
        if feat in BINARY_FEATURES:
            samples = (samples > 0.5).astype(float)

        rows[feat] = samples

    return pd.DataFrame(rows)


# ── Demo: generate baseline cohort ───────────────────────────────────────────
N_SIMULATIONS = 10_000
PATIENT        = PatientProfile()     # ← Change this to your real measurements

baseline_cohort = generate_cohort(PATIENT, n=N_SIMULATIONS)
print(f"✅  Baseline cohort shape: {baseline_cohort.shape}")
print()
baseline_cohort[CARDIAC_FEATURES].describe().round(2)

---
## 💉 Section 4 — What-If Scenario Definitions

Each **scenario** is a named dictionary of *delta functions* that modify a copy of the cohort.  
Deltas can be absolute offsets, percentage changes, or hard overrides.

In [ ]:
# ── 5. Scenario Definitions ───────────────────────────────────────────────────

@dataclass
class Scenario:
    name:        str
    description: str
    color:       str
    # Each key is a feature name, value is a callable(array) -> array
    deltas:      Dict[str, object] = field(default_factory=dict)


def apply_scenario(cohort: pd.DataFrame, scenario: Scenario) -> pd.DataFrame:
    """Return a modified copy of the cohort with scenario deltas applied."""
    df = cohort.copy()
    for feat, fn in scenario.deltas.items():
        if feat in df.columns:
            lo, hi = NHANES_PRIORS[feat][2], NHANES_PRIORS[feat][3]
            df[feat] = np.clip(fn(df[feat].values), lo, hi)
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  Define your what-if scenarios below
# ─────────────────────────────────────────────────────────────────────────────
SCENARIOS: List[Scenario] = [

    Scenario(
        name        = "Current Trajectory",
        description = "No changes — baseline measured values.",
        color       = PALETTE["baseline"],
        deltas      = {},    # <-- no modifications
    ),

    Scenario(
        name        = "Run 3× / Week",
        description = "Adds ≈9 MET-hrs/week of aerobic exercise. "
                      "Reduces systolic BP by ~5 mmHg, BMI by ~0.8, "
                      "heart rate by ~4 bpm (Framingham effect sizes).",
        color       = PALETTE["best"],
        deltas      = {
            "physical_activity_met": lambda x: x + 9.0,
            "sysBP":                 lambda x: x - 5.0,
            "bmi":                   lambda x: x - 0.8,
            "heartRate":             lambda x: x - 4.0,
            "glucose":               lambda x: x - 8.0,
            "totChol":               lambda x: x - 6.0,
            "HDLChol":               lambda x: x + 4.0,
        },
    ),

    Scenario(
        name        = "Cut Sodium to 2,300 mg/day",
        description = "Reduces sodium intake to recommended DASH-diet target. "
                      "Reduces systolic BP by ~4-6 mmHg.",
        color       = "#3498db",
        deltas      = {
            "sodium_mg_per_day": lambda x: np.minimum(x, 2300.0),
            "sysBP":             lambda x: x - 5.5,
            "diaBP":             lambda x: x - 2.5,
        },
    ),

    Scenario(
        name        = "Sleep 8 hrs / Night",
        description = "Achieves recommended 8 hrs. Linked to -3 mmHg SBP, "
                      "-5 mg/dL glucose (NHANES observational).",
        color       = "#9b59b6",
        deltas      = {
            "sleep_hours": lambda x: np.maximum(x, 8.0),
            "sysBP":       lambda x: x - 3.0,
            "glucose":     lambda x: x - 5.0,
        },
    ),

    Scenario(
        name        = "Full Lifestyle Optimisation",
        description = "Combines exercise + reduced sodium + optimal sleep.",
        color       = "#2ecc71",
        deltas      = {
            "physical_activity_met": lambda x: x + 9.0,
            "sodium_mg_per_day":     lambda x: np.minimum(x, 2300.0),
            "sleep_hours":           lambda x: np.maximum(x, 8.0),
            "sysBP":                 lambda x: x - 13.0,
            "diaBP":                 lambda x: x - 5.0,
            "bmi":                   lambda x: x - 0.8,
            "heartRate":             lambda x: x - 4.0,
            "glucose":               lambda x: x - 13.0,
            "totChol":               lambda x: x - 6.0,
            "HDLChol":               lambda x: x + 4.0,
        },
    ),

    Scenario(
        name        = "No Sodium Reduction",
        description = "Sodium increases by +500 mg/day (⚠️ worst-case dietary drift).",
        color       = PALETTE["worst"],
        deltas      = {
            "sodium_mg_per_day": lambda x: x + 500.0,
            "sysBP":             lambda x: x + 4.0,
            "diaBP":             lambda x: x + 2.0,
        },
    ),

    Scenario(
        name        = "Resume Smoking (10 cigs/day)",
        description = "Simulates relapse to light smoking. "
                      "Raises cardiac risk substantially.",
        color       = "#c0392b",
        deltas      = {
            "currentSmoker":    lambda x: np.ones_like(x),
            "cigarettes_per_day": lambda x: np.maximum(x, 10.0),
            "sysBP":            lambda x: x + 7.0,
            "totChol":          lambda x: x + 12.0,
            "HDLChol":          lambda x: x - 5.0,
        },
    ),
]

print(f"✅  {len(SCENARIOS)} scenarios defined:")
for s in SCENARIOS:
    print(f"   • {s.name}")

---
## Section 5 -- Monte Carlo Runner

For each scenario we run **10,000 virtual patients** through **all 5 predictive models**,
collecting the probability of a positive prediction across the cohort.
The spread gives us confidence intervals for cardiac, diabetes, burnout, respiratory, and kidney risk.


In [ ]:
# -- 6. Monte Carlo Runner ----------------------------------------------------

def _safe_predict_proba(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, 'predict'):
        return model.predict(X).astype(float)
    else:
        raise RuntimeError('Model has neither predict_proba nor predict.')


def run_monte_carlo(
    scenarios,
    baseline,
    model_registry,
):
    records = []
    for sc in tqdm(scenarios, desc='Running scenarios'):
        cohort = apply_scenario(baseline, sc)
        for label, model, feats in model_registry:
            avail = [f for f in feats if f in cohort.columns]
            try:
                X     = cohort[avail].values.astype(float)
                probs = _safe_predict_proba(model, X)
            except Exception as e:
                print(f'  WARN {label} error for {sc.name!r}: {e}')
                probs = np.zeros(len(cohort))

            try:
                X_base    = baseline[avail].values.astype(float)
                base_mean = float(np.mean(_safe_predict_proba(model, X_base)))
            except Exception:
                base_mean = float(np.mean(probs))

            records.append({
                'scenario':       sc.name,
                'risk_type':      label,
                'color':          sc.color,
                'mean_risk':      float(np.mean(probs)),
                'p05':            float(np.percentile(probs, 5)),
                'p25':            float(np.percentile(probs, 25)),
                'p50':            float(np.percentile(probs, 50)),
                'p75':            float(np.percentile(probs, 75)),
                'p95':            float(np.percentile(probs, 95)),
                'prob_drop_5pct': float(np.mean(probs < (base_mean - 0.05))),
                'raw_probs':      probs,
            })

    return pd.DataFrame(records)


results = run_monte_carlo(SCENARIOS, baseline_cohort, MODEL_REGISTRY)

print('\nOK  Monte Carlo complete. Summary:')
display_cols = ['scenario', 'risk_type', 'mean_risk', 'p05', 'p50', 'p95']
print(results[display_cols].to_string(index=False))

---
## 📊 Section 6 — Visualisations

In [ ]:
# -- 7. Plot A: Risk Probability Distribution per Scenario (All 5 Models) ----

RISK_LABELS = {
    'cardiac':     'Cardiac Arrest',
    'diabetes':    'Diabetes',
    'burnout':     'Burnout',
    'respiratory': 'Respiratory',
    'kidney':      'Kidney Stones',
}

def plot_risk_distributions(results, risk_type):
    df   = results[results['risk_type'] == risk_type].reset_index(drop=True)
    n_sc = len(df)
    fig, axes = plt.subplots(1, n_sc, figsize=(4.2 * n_sc, 4.5), sharey=False)
    if n_sc == 1:
        axes = [axes]
    fig.suptitle(
        f"{RISK_LABELS.get(risk_type, risk_type)} Risk Distribution -- Monte Carlo Scenarios",
        fontsize=15, fontweight='bold', y=1.02
    )
    for ax, (_, row) in zip(axes, df.iterrows()):
        probs = row['raw_probs']
        color = row['color']
        x   = np.linspace(0, 1, 500)
        kde = stats.gaussian_kde(probs, bw_method=0.15)
        y   = kde(x)
        ax.fill_between(x, y, alpha=0.3, color=color)
        ax.plot(x, y, color=color, linewidth=2)
        mask = (x >= row['p05']) & (x <= row['p95'])
        ax.fill_between(x[mask], y[mask], alpha=0.15, color=color)
        ax.axvline(row['mean_risk'], color=color, linestyle='--', linewidth=1.5)
        ax.text(row['mean_risk'], ax.get_ylim()[1] * 0.6,
                f"mu={row['mean_risk']:.2%}", ha='center', fontsize=8, color=color)
        ax.set_title(row['scenario'], fontsize=9, fontweight='bold')
        ax.set_xlabel('Predicted Risk Probability', fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.grid(True, alpha=0.2)
    fig.tight_layout()
    plt.show()


for rt in [label for label, _, _ in MODEL_REGISTRY]:
    plot_risk_distributions(results, rt)

In [ ]:
# -- 8. Plot B: Scenario Comparison Bar Chart (90% CI) -- All 5 Models -------

def plot_scenario_comparison_all(results):
    risk_types = [r for r in RISK_LABELS if r in results['risk_type'].unique()]
    n     = len(risk_types)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    fig.suptitle(
        'What-If Scenario Comparison -- Mean Risk with 90% CI (All 5 Models)',
        fontsize=14, fontweight='bold'
    )
    for ax, risk_type in zip(axes_flat, risk_types):
        df    = results[results['risk_type'] == risk_type].reset_index(drop=True)
        y_pos = np.arange(len(df))
        colors = df['color'].tolist()
        ax.barh(y_pos, df['mean_risk'], color=colors, alpha=0.8,
                edgecolor='white', linewidth=0.5, height=0.55)
        err_lo = df['mean_risk'] - df['p05']
        err_hi = df['p95']       - df['mean_risk']
        ax.errorbar(df['mean_risk'], y_pos, xerr=[err_lo, err_hi],
                    fmt='none', color='white', alpha=0.5, capsize=4, linewidth=1.2)
        for i, (_, row) in enumerate(df.iterrows()):
            ax.text(row['mean_risk'] + 0.003, i,
                    f"{row['mean_risk']:.1%}",
                    va='center', ha='left', fontsize=9, color='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df['scenario'], fontsize=9)
        ax.set_xlabel('Predicted Risk Probability')
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_title(RISK_LABELS.get(risk_type, risk_type), fontsize=12, fontweight='bold')
        ax.grid(True, axis='x', alpha=0.2)
        ax.invert_yaxis()
    for ax in axes_flat[len(risk_types):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()


plot_scenario_comparison_all(results)

In [ ]:
# -- 9. Plot C: 6-Month Trajectory Projection (All 5 Models) -----------------
#
# Effect size ramps linearly from 0% (week 0) to 100% (week 24).

def project_trajectory(
    scenario, baseline, model, features,
    n_weeks=24, n_boot=2_000, seed=99,
):
    rng   = np.random.default_rng(seed)
    avail = [f for f in features if f in baseline.columns]
    idx   = rng.choice(len(baseline), size=min(n_boot, len(baseline)), replace=False)
    sub   = baseline.iloc[idx].copy()
    rows  = []
    for week in range(n_weeks + 1):
        frac   = week / n_weeks
        cohort = sub.copy()
        for feat, fn in scenario.deltas.items():
            if feat in cohort.columns:
                original = sub[feat].values
                target   = np.clip(fn(original),
                                   NHANES_PRIORS[feat][2],
                                   NHANES_PRIORS[feat][3])
                cohort[feat] = original + frac * (target - original)
        X     = cohort[avail].values.astype(float)
        probs = _safe_predict_proba(model, X)
        rows.append({'week': week, 'mean': np.mean(probs),
                     'p10': np.percentile(probs, 10),
                     'p25': np.percentile(probs, 25),
                     'p75': np.percentile(probs, 75),
                     'p90': np.percentile(probs, 90)})
    return pd.DataFrame(rows)


TRAJECTORY_SCENARIOS = [SCENARIOS[0], SCENARIOS[1], SCENARIOS[4], SCENARIOS[5]]

n_models = len(MODEL_REGISTRY)
ncols = min(3, n_models)
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5 * nrows))
axes_flat = np.array(axes).flatten() if n_models > 1 else [axes]
fig.suptitle('6-Month Risk Trajectory -- Lifestyle Change Ramp-Up (All 5 Models)',
             fontsize=13, fontweight='bold')

for ax, (label, model, feats) in zip(axes_flat, MODEL_REGISTRY):
    ax.set_title(RISK_LABELS.get(label, label), fontsize=11, fontweight='bold')
    for sc in tqdm(TRAJECTORY_SCENARIOS, desc=f'Projecting {label}', leave=False):
        traj  = project_trajectory(sc, baseline_cohort, model, feats)
        weeks = traj['week']
        ax.fill_between(weeks, traj['p10'], traj['p90'], alpha=0.10, color=sc.color)
        ax.fill_between(weeks, traj['p25'], traj['p75'], alpha=0.18, color=sc.color)
        ax.plot(weeks, traj['mean'], color=sc.color, linewidth=2.2, label=sc.name)
    ax.set_xlabel('Week', fontsize=10)
    ax.set_ylabel('Mean Predicted Risk', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))
    ax.set_xticklabels([f'Wk {w}' for w in range(0, 25, 4)])

for ax in axes_flat[n_models:]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# -- 10. Plot D: Probability Statement Dashboard (All 5 Models) ---------------

def compute_prob_statements(results, risk_type, threshold=0.05):
    base_row = results[
        (results['scenario'] == 'Current Trajectory') &
        (results['risk_type'] == risk_type)
    ]
    if base_row.empty:
        return pd.DataFrame()
    base_row   = base_row.iloc[0]
    base_mean  = np.mean(base_row['raw_probs'])
    rows = []
    for _, row in results[results['risk_type'] == risk_type].iterrows():
        sc_probs = row['raw_probs']
        rows.append({
            'Scenario':                    row['scenario'],
            f'P(drops>={threshold:.0%})':  float(np.mean(sc_probs < (base_mean - threshold))),
            f'P(rises>={threshold:.0%})':  float(np.mean(sc_probs > (base_mean + threshold))),
            'Mean Risk':                   row['mean_risk'],
            'Delta vs Baseline':           row['mean_risk'] - base_mean,
        })
    return pd.DataFrame(rows)


for rt in [label for label, _, _ in MODEL_REGISTRY]:
    print(f"\n{'='*70}")
    print(f"  {RISK_LABELS.get(rt, rt)} Risk -- Probability Statements (threshold=5%)")
    print(f"{'='*70}")
    df_stmt = compute_prob_statements(results, rt, threshold=0.05)
    if not df_stmt.empty:
        print(df_stmt.to_string(index=False, float_format='{:.1%}'.format))

In [ ]:
# -- 11. Plot E: Feature Sensitivity -- Tornado Analysis (All 5 Models) ------

def feature_sensitivity(baseline, model, features, n_boot=3_000, seed=77):
    rng  = np.random.default_rng(seed)
    avail = [f for f in features if f in baseline.columns]
    idx  = rng.choice(len(baseline), size=min(n_boot, len(baseline)), replace=False)
    sub  = baseline[avail].iloc[idx].values.astype(float)
    base_risk = np.mean(_safe_predict_proba(model, sub))
    deltas = {}
    for i, feat in enumerate(avail):
        sd  = NHANES_PRIORS.get(feat, (0, 1, 0, 1))[1]
        hi  = sub.copy(); hi[:, i] += sd
        lo  = sub.copy(); lo[:, i] -= sd
        deltas[feat] = (
            np.mean(_safe_predict_proba(model, hi)) -
            np.mean(_safe_predict_proba(model, lo))
        ) / (2 * sd + 1e-9)
    return pd.Series(deltas).sort_values(ascending=False)


CMAPS = ['RdPu', 'BuPu', 'YlOrBr', 'GnBu', 'PuRd']
n_models = len(MODEL_REGISTRY)
ncols = min(3, n_models)
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
axes_flat = np.array(axes).flatten() if n_models > 1 else [axes]
fig.suptitle('Feature Sensitivity -- Tornado Analysis (+-1 SD, All 5 Models)',
             fontsize=13, fontweight='bold')

for ax, (label, model, feats), cmap in zip(axes_flat, MODEL_REGISTRY, CMAPS):
    sens      = feature_sensitivity(baseline_cohort, model, feats)
    cfn       = plt.cm.get_cmap(cmap)
    vals      = sens.values
    norm_vals = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    bar_colors = [cfn(v * 0.7 + 0.2) for v in norm_vals]
    ax.barh(sens.index[::-1], sens.values[::-1],
            color=bar_colors[::-1], alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='white', linewidth=0.8, alpha=0.5)
    ax.set_title(RISK_LABELS.get(label, label), fontsize=11, fontweight='bold')
    ax.set_xlabel('DeltaRisk per unit change', fontsize=9)
    ax.grid(True, axis='x', alpha=0.2)

for ax in axes_flat[n_models:]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# -- 12. Plot F: Violin Plot -- All 5 Risk Models ----------------------------

def plot_violin_all(results):
    risk_types = [r for r in RISK_LABELS if r in results['risk_type'].unique()]
    n    = len(risk_types)
    fig, axes = plt.subplots(n, 1, figsize=(16, 5 * n))
    if n == 1:
        axes = [axes]
    for ax, risk_type in zip(axes, risk_types):
        df     = results[results['risk_type'] == risk_type].reset_index(drop=True)
        labels = df['scenario'].tolist()
        data   = [row['raw_probs'] * 100 for _, row in df.iterrows()]
        colors = df['color'].tolist()
        ax.set_title(
            f"{RISK_LABELS.get(risk_type, risk_type)} Risk -- Violin Distribution per Scenario",
            fontsize=12, fontweight='bold'
        )
        parts = ax.violinplot(data, positions=range(len(data)),
                              showmeans=True, showmedians=True, widths=0.7)
        for body, color in zip(parts['bodies'], colors):
            body.set_facecolor(color)
            body.set_alpha(0.55)
        for key in ['cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes']:
            if key in parts:
                parts[key].set_color('white')
                parts[key].set_linewidth(1.2)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
        ax.set_ylabel('Predicted Risk (%)', fontsize=10)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
        ax.grid(True, axis='y', alpha=0.2)
    fig.tight_layout()
    plt.show()


plot_violin_all(results)

---
## 📝 Section 7 — Clinical Summary & Interpretation

In [ ]:
# -- 13. Auto-generated Clinical Narrative (All 5 Models) --------------------

def generate_summary(results):
    lines = ['\n' + '=' * 72]
    lines.append('  TSUKUMO -- WHAT-IF SCENARIO CLINICAL SUMMARY (5 MODELS)')
    lines.append('=' * 72)

    risk_types = [label for label, _, _ in MODEL_REGISTRY]
    base_risks = {}
    for rt in risk_types:
        row = results[
            (results['scenario'] == 'Current Trajectory') &
            (results['risk_type'] == rt)
        ]
        if not row.empty:
            base_risks[rt] = row.iloc[0]

    lines.append(
        f'\nBASELINE PATIENT\n'
        f'  Age: {PATIENT.age:.0f}  |  BMI: {PATIENT.bmi:.1f}  |  SBP: {PATIENT.sysBP:.0f} mmHg\n'
        f'  Glucose: {PATIENT.glucose:.0f} mg/dL  |  TotChol: {PATIENT.totChol:.0f} mg/dL\n'
        f'\nBASELINE RISK (Monte Carlo, N={N_SIMULATIONS:,})'
    )
    for rt, row in base_risks.items():
        lines.append(
            f"  {RISK_LABELS.get(rt, rt):18s}: {row['mean_risk']:.1%}"
            f"  (90% CI: {row['p05']:.1%} - {row['p95']:.1%})"
        )

    lines.append('\nSCENARIO RESULTS')
    lines.append('-' * 72)
    for sc in SCENARIOS[1:]:
        lines.append(f'\n  {sc.name}')
        lines.append(f'     {sc.description}')
        for rt in risk_types:
            sc_data = results[
                (results['scenario'] == sc.name) & (results['risk_type'] == rt)
            ]
            br = base_risks.get(rt)
            if sc_data.empty or br is None:
                continue
            sc_row = sc_data.iloc[0]
            delta  = sc_row['mean_risk'] - br['mean_risk']
            arrow  = 'down' if delta < 0 else 'up'
            lines.append(
                f"     {RISK_LABELS.get(rt, rt):18s}: {sc_row['mean_risk']:.1%}"
                f"  ({arrow} {delta:+.1%} vs baseline)"
                f"  90% CI [{sc_row['p05']:.1%}, {sc_row['p95']:.1%}]"
            )

    lines.append('\n' + '=' * 72)
    lines.append('  DISCLAIMER: Statistical simulation only -- not medical advice.')
    lines.append('  Consult a licensed physician before making clinical decisions.')
    lines.append('=' * 72)
    return '\n'.join(lines)


print(generate_summary(results))

---
## 🔧 Section 8 — Interactive Custom Scenario Builder

Use the sliders below to define your own what-if scenario in real time.

In [ ]:
# -- 14. Interactive Widget (All 5 Models) ------------------------------------
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    style  = {'description_width': '200px'}
    layout = widgets.Layout(width='500px')

    w_sysbp  = widgets.IntSlider(value=int(PATIENT.sysBP),  min=80,  max=220,  step=1,   description='Systolic BP (mmHg)',        style=style, layout=layout)
    w_bmi    = widgets.FloatSlider(value=PATIENT.bmi,        min=15,  max=60,   step=0.5, description='BMI',                       style=style, layout=layout)
    w_glu    = widgets.IntSlider(value=int(PATIENT.glucose), min=60,  max=400,  step=1,   description='Glucose (mg/dL)',           style=style, layout=layout)
    w_sleep  = widgets.FloatSlider(value=PATIENT.sleep_hours,min=3,   max=12,   step=0.5, description='Sleep Hours',               style=style, layout=layout)
    w_met    = widgets.FloatSlider(value=PATIENT.physical_activity_met, min=1, max=15, step=0.5, description='Physical Activity (MET)', style=style, layout=layout)
    w_sodium = widgets.IntSlider(value=int(PATIENT.sodium_mg_per_day), min=500, max=8000, step=100, description='Sodium (mg/day)',      style=style, layout=layout)
    w_chol   = widgets.IntSlider(value=int(PATIENT.totChol), min=100, max=400,  step=5,   description='Total Cholesterol (mg/dL)',  style=style, layout=layout)
    btn      = widgets.Button(description='Run My Scenario', button_style='info', icon='play',
                               layout=widgets.Layout(width='200px', margin='12px 0px'))
    out      = widgets.Output()

    def on_run(_):
        with out:
            clear_output(wait=True)
            custom_patient = PatientProfile(
                sysBP               = w_sysbp.value,
                bmi                 = w_bmi.value,
                glucose             = w_glu.value,
                sleep_hours         = w_sleep.value,
                physical_activity_met = w_met.value,
                sodium_mg_per_day   = w_sodium.value,
                totChol             = w_chol.value,
                age                 = PATIENT.age,
                diaBP               = PATIENT.diaBP,
                heartRate           = PATIENT.heartRate,
                HDLChol             = PATIENT.HDLChol,
                cigarettes_per_day  = PATIENT.cigarettes_per_day,
                alcohol_drinks_per_week = PATIENT.alcohol_drinks_per_week,
                prevalentHyp        = PATIENT.prevalentHyp,
                diabetes            = PATIENT.diabetes,
                currentSmoker       = PATIENT.currentSmoker,
            )
            cohort = generate_cohort(custom_patient, n=5_000)

            risk_results = {}
            for label, model, feats in MODEL_REGISTRY:
                avail = [f for f in feats if f in cohort.columns]
                X = cohort[avail].values.astype(float)
                risk_results[label] = _safe_predict_proba(model, X)

            print('=' * 58)
            print('  YOUR CUSTOM WHAT-IF SCENARIO RESULT (5 MODELS)')
            print('=' * 58)
            print(f'  SBP={w_sysbp.value} | BMI={w_bmi.value:.1f} | Glucose={w_glu.value}')
            print(f'  Sleep={w_sleep.value}h | MET={w_met.value:.1f} | Na={w_sodium.value}mg')
            print('-' * 58)
            for label, probs in risk_results.items():
                mu    = np.mean(probs)
                ci_lo = np.percentile(probs, 5)
                ci_hi = np.percentile(probs, 95)
                print(f'  {RISK_LABELS.get(label, label):18s}: {mu:.1%}  (90% CI: {ci_lo:.1%}-{ci_hi:.1%})')
            print('=' * 58)

            fig, ax = plt.subplots(figsize=(10, 4))
            lbl_plot   = [RISK_LABELS.get(l, l) for l in risk_results]
            means_plot = [float(np.mean(p)) * 100 for p in risk_results.values()]
            bar_clrs   = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6']
            bars = ax.bar(lbl_plot, means_plot, color=bar_clrs[:len(lbl_plot)], alpha=0.82, width=0.5)
            for bar, r in zip(bars, [v / 100 for v in means_plot]):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                        f'{r:.1%}', ha='center', fontsize=10, fontweight='bold', color='white')
            ax.set_ylabel('Predicted Risk (%)')
            ax.set_title('Your Custom Scenario -- All 5 Predicted Risks', fontsize=11)
            ax.set_ylim(0, max(means_plot) * 1.4 if means_plot else 10)
            ax.grid(True, axis='y', alpha=0.2)
            fig.tight_layout()
            plt.show()

    btn.on_click(on_run)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color:#7b7bff'>Custom What-If Scenario Builder (All 5 Models)</h3>"),
        widgets.HTML("<p style='color:#aaa'>Adjust the sliders, then click Run My Scenario</p>"),
        w_sysbp, w_bmi, w_glu, w_sleep, w_met, w_sodium, w_chol,
        btn, out,
    ]))

except ImportError:
    print('ipywidgets not available -- run: pip install ipywidgets')
    print('Then re-execute this cell for the interactive slider UI.')

---
## 💾 Section 9 — Export Results

In [ ]:
# ── 15. Export to CSV ─────────────────────────────────────────────────────────
export_cols = ["scenario", "risk_type", "mean_risk", "p05", "p25", "p50", "p75", "p95"]
export_df   = results[export_cols].copy()
export_df.to_csv("whatif_results.csv", index=False)
print("✅  Exported to whatif_results.csv")

# ── Clinical summary text ─────────────────────────────────────────────────────
with open("whatif_summary.txt", "w", encoding="utf-8") as f:
    f.write(generate_summary(results))
print("✅  Clinical summary written to whatif_summary.txt")

print("\nAll outputs saved in:", os.path.abspath("."))